# Concrete Crack and Spalling Segmentation with YOLOv8

## Environment Setup

This section ensures that all necessary libraries are installed and the GPU environment is ready for training. It includes installing `ultralytics`, `roboflow`, and `opencv-python-headless`.

In [1]:
!nvidia-smi
!pip install ultralytics roboflow opencv-python-headless

Sun Jun  7 01:23:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Data Acquisition

This section handles the download of the concrete crack and spalling dataset from Roboflow. The dataset is configured for YOLOv8 segmentation tasks.

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="gmwTsWC1x1sthZSCdXDo")
project = rf.workspace("1-7frt7").project("concrete-crack-spalling")
version = project.version(5)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to concrete-crack-&-spalling-5 in yolov8:: 100%|██████████| 7131/7131 [00:01<00:00, 4149.33it/s]


In [5]:
print(dataset.location)

/content/concrete-crack-&-spalling-5


In [7]:
dataset_path = "/content/concrete-crack-&-spalling-5"

In [8]:
!find "{dataset_path}" -maxdepth 2 -type d

/content/concrete-crack-&-spalling-5
/content/concrete-crack-&-spalling-5/train
/content/concrete-crack-&-spalling-5/train/images
/content/concrete-crack-&-spalling-5/train/labels
/content/concrete-crack-&-spalling-5/valid
/content/concrete-crack-&-spalling-5/valid/images
/content/concrete-crack-&-spalling-5/valid/labels


In [9]:
!cat "{dataset_path}/data.yaml"

names:
- crack
- spalling
nc: 2
roboflow:
  license: CC BY 4.0
  project: concrete-crack-spalling
  url: https://universe.roboflow.com/1-7frt7/concrete-crack-spalling/dataset/5
  version: 5
  workspace: 1-7frt7
test: ../test/images
train: ../train/images
val: ../valid/images


## Model Training and Evaluation

Here, a YOLOv8n-seg model is initialized and trained on the downloaded dataset. After training, the model's performance is validated using the validation split of the dataset.

In [11]:
from ultralytics import YOLO

dataset_path = "/content/concrete-crack-&-spalling-5"

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data=f"{dataset_path}/data.yaml",
    imgsz=640,
    epochs=50,
    batch=8,
    patience=10,
    project="/content/runs/infra_defect_seg",
    name="yolov8n_seg_crack_spalling"
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/concrete-crack-&-spalling-5/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_crack_spalling, nbs=64, nms=False, opset=None, optimize=False, optimiz

In [12]:
best_model = YOLO("/content/runs/infra_defect_seg/yolov8n_seg_crack_spalling/weights/best.pt")

metrics = best_model.val(
    data=f"{dataset_path}/data.yaml",
    split="val",
    imgsz=640,
    conf=0.25,
    iou=0.45
)

print(metrics)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2020.0±1251.5 MB/s, size: 73.0 KB)
val: Scanning /content/concrete-crack-&-spalling-5/valid/labels.cache... 300 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.4s
                   all        300        361      0.936      0.894      0.895      0.769      0.936      0.894      0.891      0.601
                 crack        223        255      0.935      0.863      0.865      0.733      0.935      0.863      0.857      0.422
              spalling         78        106      0.937      0.925      0.925      0.805      0.937      0.925      0.925      0.779
Speed: 3.9ms preprocess

## Inference

This section performs inference using the best-trained model on the validation images. The predictions, including bounding boxes and segmentation masks, are saved.

In [14]:
best_model.predict(
    source=f"{dataset_path}/valid/images",
    save=True,
    save_txt=True,
    conf=0.25,
    project="/content/predictions",
    name="valid_outputs"
)


image 1/300 /content/concrete-crack-&-spalling-5/valid/images/-1104-_jpg.rf.827c4914aa8144fe2b9c4e7a52694c13.jpg: 640x640 2 spallings, 14.9ms
image 2/300 /content/concrete-crack-&-spalling-5/valid/images/-990-_jpg.rf.59f1888c28c7f1f145eb58562549703b.jpg: 640x640 2 spallings, 14.5ms
image 3/300 /content/concrete-crack-&-spalling-5/valid/images/000133_jpg.rf.dd30a052d958211196d9a920dfaaac7b.jpg: 640x640 1 spalling, 9.9ms
image 4/300 /content/concrete-crack-&-spalling-5/valid/images/00021_jpg.rf.fca8c47f62604421c48970f9b8f68d7a.jpg: 640x640 1 crack, 13.0ms
image 5/300 /content/concrete-crack-&-spalling-5/valid/images/001135_jpg.rf.affd7a1ac9f9de804d8780b5d5ecbfc2.jpg: 640x640 1 spalling, 17.0ms
image 6/300 /content/concrete-crack-&-spalling-5/valid/images/001136_jpg.rf.14bf4ff261dc5ad454151087439e90cd.jpg: 640x640 1 spalling, 9.7ms
image 7/300 /content/concrete-crack-&-spalling-5/valid/images/001141_jpg.rf.c0a682b60c98f11af7117566ea63cb9d.jpg: 640x640 1 spalling, 9.1ms
image 8/300 /conte

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'crack', 1: 'spalling'}
 obb: None
 orig_img: array([[[161, 162, 166],
         [173, 174, 178],
         [179, 180, 184],
         ...,
         [209, 202, 207],
         [214, 207, 212],
         [221, 214, 219]],
 
        [[161, 162, 166],
         [172, 173, 177],
         [179, 180, 184],
         ...,
         [206, 199, 204],
         [209, 202, 207],
         [214, 207, 212]],
 
        [[164, 165, 169],
         [172, 173, 177],
         [177, 178, 182],
         ...,
         [205, 198, 201],
         [207, 200, 203],
         [211, 204, 207]],
 
        ...,
 
        [[198, 186, 182],
         [194, 182, 178],
         [210, 198, 194],
         ...,
         [183, 188, 189],
         [179, 184, 187],
         [171, 176, 179]],
 
        [[194, 182, 178],
         [195, 183, 179],
         

## Download Results and Sample Images

Finally, the trained model's weights, the generated predictions, and a sample of the input images are prepared for download for further analysis or deployment.

In [15]:
!zip -r /content/valid_predictions.zip /content/predictions/valid_outputs

  adding: content/predictions/valid_outputs/ (stored 0%)
  adding: content/predictions/valid_outputs/c3aa1d12ep4583dee092c7d423178be7_JPG.rf.296d196bf92e9fcae7398742c0d68fba.jpg (deflated 6%)
  adding: content/predictions/valid_outputs/2398_jpg.rf.ef1008de09db87ed2719559a21d0f730.jpg (deflated 10%)
  adding: content/predictions/valid_outputs/2305_jpg.rf.bf21a1d0976f165433409396710f6f5c.jpg (deflated 10%)
  adding: content/predictions/valid_outputs/001306_jpg.rf.10f2398632e011cc2092c1e13cc23dc1.jpg (deflated 5%)
  adding: content/predictions/valid_outputs/train_0466_jpg.rf.c7ebc7dce30091cb67539993bcbd508e.jpg (deflated 6%)
  adding: content/predictions/valid_outputs/2523_jpg.rf.968bf1b1754277c1bfa7f2d4a6d4fcce.jpg (deflated 7%)
  adding: content/predictions/valid_outputs/Grieta_0057_jpg.rf.f0a4bbf25d422ae22791ccc50c417258.jpg (deflated 6%)
  adding: content/predictions/valid_outputs/2141_jpg.rf.3dedeece9b4baef7d5807e2f0851a0ea.jpg (deflated 7%)
  adding: content/predictions/valid_output

In [16]:
from google.colab import files

files.download("/content/runs/infra_defect_seg/yolov8n_seg_crack_spalling/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!mkdir -p /content/demo_images
!cp "/content/concrete-crack-&-spalling-5/valid/images/"*.jpg /content/demo_images/ 2>/dev/null || true
!cp "/content/concrete-crack-&-spalling-5/valid/images/"*.png /content/demo_images/ 2>/dev/null || true
!ls /content/demo_images | head

000133_jpg.rf.dd30a052d958211196d9a920dfaaac7b.jpg
00021_jpg.rf.fca8c47f62604421c48970f9b8f68d7a.jpg
001135_jpg.rf.affd7a1ac9f9de804d8780b5d5ecbfc2.jpg
001136_jpg.rf.14bf4ff261dc5ad454151087439e90cd.jpg
001141_jpg.rf.c0a682b60c98f11af7117566ea63cb9d.jpg
001216_jpg.rf.641328967badded4b2601f1412669e61.jpg
001222_jpg.rf.52e8c706b148b1255b6c702a8a269b25.jpg
001232_jpg.rf.91cb7b3a18d369d26e662c2b74384e24.jpg
001290_jpg.rf.a02924230686dd2dfb755d0c17863b47.jpg
001303_jpg.rf.652e9d5ca87ea36bf81af35de8668f7d.jpg


In [18]:
!mkdir -p /content/demo_sample
!find "/content/concrete-crack-&-spalling-5/valid/images" -type f | head -10 | xargs -I {} cp "{}" /content/demo_sample/
!zip -r /content/demo_sample.zip /content/demo_sample

  adding: content/demo_sample/ (stored 0%)
  adding: content/demo_sample/c3aa1d12ep4583dee092c7d423178be7_JPG.rf.296d196bf92e9fcae7398742c0d68fba.jpg (deflated 0%)
  adding: content/demo_sample/2398_jpg.rf.ef1008de09db87ed2719559a21d0f730.jpg (deflated 3%)
  adding: content/demo_sample/2305_jpg.rf.bf21a1d0976f165433409396710f6f5c.jpg (deflated 6%)
  adding: content/demo_sample/001306_jpg.rf.10f2398632e011cc2092c1e13cc23dc1.jpg (deflated 1%)
  adding: content/demo_sample/train_0466_jpg.rf.c7ebc7dce30091cb67539993bcbd508e.jpg (deflated 3%)
  adding: content/demo_sample/2523_jpg.rf.968bf1b1754277c1bfa7f2d4a6d4fcce.jpg (deflated 9%)
  adding: content/demo_sample/Grieta_0057_jpg.rf.f0a4bbf25d422ae22791ccc50c417258.jpg (deflated 2%)
  adding: content/demo_sample/2141_jpg.rf.3dedeece9b4baef7d5807e2f0851a0ea.jpg (deflated 3%)
  adding: content/demo_sample/2576_jpg.rf.935855ca388ffddcb24380e539047b68.jpg (deflated 9%)
  adding: content/demo_sample/train_0429_jpg.rf.a70cb16aeace7cdf2ef459f431e9d

In [19]:
from google.colab import files
files.download("/content/demo_sample.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>